In [2]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [3]:
%pip install scikit-learn

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

Note: you may need to restart the kernel to use updated packages.


- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [10]:
data = pd.read_csv("../data/kg_train.csv", encoding="latin-1")

print(data.shape)

(5964, 2)


In [7]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv", encoding="latin-1")

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [17]:
from sklearn.model_selection import train_test_split

data_train, data_val = train_test_split(data,test_size=0.2,random_state=42)

print(data.shape)
print(data_train.shape)
print(data_val.shape)

(5964, 2)
(4771, 2)
(1193, 2)


## Data Preprocessing

In [18]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [19]:
import re

In [20]:
def clean_html(text):

    text = re.sub(r'<script.*?>.*?</script>', '', text, flags=re.DOTALL)
    text = re.sub(r'<style.*?>.*?</style>', '', text, flags=re.DOTALL)

    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)

    text = re.sub(r'<.*?>', '', text)

    return text

In [21]:
data_train['text'] = data_train['text'].apply(clean_html)
data_val['text'] = data_val['text'].apply(clean_html)

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [22]:
def clean_text(text):

    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    text = re.sub(r'\d+', ' ', text)

    text = re.sub(r'\b[a-zA-Z]\b', ' ', text)

    text = re.sub(r'^\s*[a-zA-Z]\s+', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    text = re.sub(r'^b\s+', '', text)

    text = text.lower()

    return text.strip()

## Now let's work on removing stopwords
Remove the stopwords.

In [23]:
stop_words = set(stopwords.words('english'))

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [28]:
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

stop_words = set(stopwords.words("english"))
snowball = SnowballStemmer("english")

text = data_train["text"].iloc[0]

filtered_tokens = [
    snowball.stem(token.lower())
    for token in word_tokenize(text)
    if token.lower() not in stop_words and token.isalpha()
]

print("Original text:")
print(text)

print("\nFiltered Tokens:")
print(filtered_tokens)

Original text:
back -- we need to talkCall only my berry.

Filtered Tokens:
['back', 'need', 'talkcal', 'berri']


In [31]:
import nltk

from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

lemmatizer = WordNetLemmatizer()

In [32]:
from nltk.corpus import wordnet

def get_wordnet_pos(word):
            tag = nltk.pos_tag([word])[0][1][0]
            tag_dict = {
                "J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV
            }
            return tag_dict.get(tag, wordnet.NOUN)

lemmatized_tokens = []

for word in filtered_tokens:
            lemmatized_tokens.append(lemmatizer.lemmatize(word, get_wordnet_pos(word)))

print()
print('Lemmatized Tokens:', lemmatized_tokens)


Lemmatized Tokens: ['back', 'need', 'talkcal', 'berri']


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [35]:
from collections import Counter

ham_text = " ".join(data_train[data_train["label"] == 0]["text"])
spam_text = " ".join(data_train[data_train["label"] == 1]["text"])

ham_words = ham_text.split()
spam_words = spam_text.split()

top_10_ham = Counter(ham_words).most_common(10)
top_10_spam = Counter(spam_words).most_common(10)

print("Top 10 HAM:")
print(top_10_ham)

print("\nTop 10 SPAM:")
print(top_10_spam)

Top 10 HAM:
[('the', 10686), ('to', 6608), ('of', 5392), ('and', 5259), ('a', 4051), ('in', 3839), ('that', 2413), ('is', 2205), ('for', 2129), ('on', 1921)]

Top 10 SPAM:
[('the', 26451), ('to', 21801), ('of', 19124), ('and', 14683), ('I', 12339), ('in', 11971), ('you', 10533), ('this', 9009), ('a', 8805), ('for', 8022)]


In [41]:
from nltk.tokenize import word_tokenize

def preprocess_text(text):
    tokens = word_tokenize(text)

    filtered_tokens = [token.lower() for token in tokens if token.lower() not in stop_words and token.isalpha()]

    lemmatized_tokens = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in filtered_tokens]

    return " ".join(lemmatized_tokens)

In [39]:
data_train["preprocessed_text"] = data_train["text"].apply(preprocess_text)
data_val["preprocessed_text"] = data_val["text"].apply(preprocess_text)

## Extra features

In [40]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,text,label,preprocessed_text,money_mark,suspicious_words,text_len
3821,back -- we need to talkCall only my berry.,0,back need talkcall berry,0,0,24
2407,What is mullen's first name?,0,mullen first name,0,0,17
1173,Greetings from Dubai=2CThis letter must come t...,1,greeting letter must come big believe day peop...,1,1,1863
4231,Sullivan Jacob J Friday December 4 2009 3:31 A...,0,sullivan jacob j friday december amiranthe eu ...,0,1,376
1666,Fyi,0,fyi,0,0,3


## How would work the Bag of Words with Count Vectorizer concept?

In [46]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

corpus = data_train["preprocessed_text"]

vectorizer = CountVectorizer()

X_bow = vectorizer.fit_transform(corpus)

print("Shape:", X_bow.shape)
print("value number non zero:", X_bow.nnz)

Shape: (4771, 34764)
value number non zero: 315614


In [47]:
first_email = X_bow[0].toarray()[0]

words = vectorizer.get_feature_names_out()

result = pd.DataFrame({"word": words, "count": first_email})

result[result["count"] > 0].sort_values("count", ascending=False).head(20)

,word,count
3535,back,1
4313,berry,1
20311,need,1
29099,talkcall,1


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(data_train["preprocessed_text"])
X_val_tfidf = tfidf.transform(data_val["preprocessed_text"])

print("Train shape:", X_train_tfidf.shape)
print("Validation shape:", X_val_tfidf.shape)

Train shape: (4771, 34764)
Validation shape: (1193, 34764)


## And the Train a Classifier?

In [50]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

y_train = data_train["label"]
y_val = data_val["label"]

model.fit(X_train_tfidf, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[2712.,2059.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.56,-0.84]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 34764)","[[0. ,0.07,0. ,...,0. ,0. ,0. ], [0.04,0. ,0.82,...,0.03,0.03,0.03]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 34764)","[[-10.68,-10.6 ,-10.68,...,-10.68,-10.68,-10.68], [-10.84,-10.87,-10.27,...,-10.85,-10.85,-10.85]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,34764


### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
# Your code